In [1]:
import sys  
sys.path.insert(1, '/home/spuchin/GitHub/the-hand-of-midas')

from clickhouse_connect.driver.asyncclient import AsyncClient as AsyncClickHouseClient
from pandas import DataFrame
import plotly.graph_objects as go
from talib._ta_lib import EMA

from src.adapters.repositories.candlesticks import CandlesticksRepository
from src.adapters.repositories.common.clickhouse_base import get_clickhouse_client
from src.schemas.binance import BinanceIntervalEnum, BinanceSectionEnum
from src.schemas.candlesticks import CandlesticksInputSchema
from src.services.candlesticks import CandlesticksService
from src.settings import settings

In [2]:
clickhouse_client: AsyncClickHouseClient = await get_clickhouse_client()
candlesticks_repository: CandlesticksRepository = CandlesticksRepository(client=clickhouse_client)
candlesticks_service: CandlesticksService = CandlesticksService(repository=candlesticks_repository)

In [3]:
ohlc: DataFrame = await candlesticks_service.get_candlesticks(
    input_schema=CandlesticksInputSchema(
        ticker=settings.TICKER,
        exchange=settings.BINANCE_EXCHANGE_NAME,
        section=BinanceSectionEnum.BINANCE_SPOT,
        interval=BinanceIntervalEnum.FOUR_HOURS
    )
)
ohlc.sort_values(by="open_time", inplace=True)
ohlc.head()

,open,high,low,close,open_time
0,4261.48,4349.99,4261.32,4349.99,2017-08-17 04:00:00
1,4333.32,4485.39,4333.32,4427.30,2017-08-17 08:00:00
2,4436.06,4485.39,4333.42,4352.34,2017-08-17 12:00:00
3,4352.33,4354.84,4200.74,4325.23,2017-08-17 16:00:00
4,4307.56,4369.69,4258.56,4285.08,2017-08-17 20:00:00


In [4]:
EXPONENTIONAL_MOVING_AVERAGES: list[int] = [21]

for moving_average in EXPONENTIONAL_MOVING_AVERAGES:
    ohlc["open-macro"] = EMA(ohlc.open.values, moving_average)
    ohlc["high-macro"] = EMA(ohlc.high.values, moving_average)
    ohlc["low-macro"] = EMA(ohlc.low.values, moving_average)
    ohlc["close-macro"] = EMA(ohlc.close.values, moving_average)

    ohlc["open-macro"] = ohlc["open-macro"].shift(1)
    ohlc["high-macro"] = ohlc["high-macro"].shift(1)
    ohlc["low-macro"] = ohlc["low-macro"].shift(1)
    ohlc["close-macro"] = ohlc["close-macro"].shift(1)

In [5]:
HEAD: int = 1000
EXPONENTIONAL_MOVING_AVERAGE: int = 21

go.Figure(
    data=[
        go.Candlestick(
            x=ohlc["open_time"].head(HEAD),
            open=ohlc["open-macro"].head(HEAD),
            high=ohlc["high-macro"].head(HEAD),
            low=ohlc["low-macro"].head(HEAD),
            close=ohlc["close-macro"].head(HEAD)
        )
    ]
)

In [6]:
go.Figure(
    data=[
        go.Candlestick(
            x=ohlc[f"open_time"].head(HEAD),
            open=ohlc[f"open"].head(HEAD),
            high=ohlc[f"high"].head(HEAD),
            low=ohlc[f"low"].head(HEAD),
            close=ohlc[f"close"].head(HEAD)
        )
    ]
)

In [7]:
ohlc.to_csv("macro-candles.csv", index=False)